## **my_sample_01**

Drying Oven의 온도를 일정하게 유지하기 위해 PPO(Proximal Policy Optimization) 알고리즘을 사용하는 기초 프레임워크입니다. Gymnasium 라이브러리를 활용해 환경을 구축하고, Stable Baselines3로 강화학습을 구현하는 방식이 가장 효율적입니다.


### 1. Oven 시뮬레이션 환경 정의 (Gym Custom Env)

먼저 온도, 습도, 풍압, RPM 등을 상태(State)로 갖는 가상 환경을 만들어야 합니다.

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DryingOvenEnv(gym.Env):
    def __init__(self):
        super(DryingOvenEnv, self).__init__()
        
        # 제어 동작: 팬 RPM 증감, 히터 출력 조절 등 (연속적 값 -1 ~ 1)
        self.action_space = spaces.Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        
        # 관측 데이터: [내부온도, 내부습도, 배기온도, 풍압, 현재RPM]
        self.observation_space = spaces.Box(low=0, high=200, shape=(5,), dtype=np.float32)
        
        self.target_temp = 80.0  # 목표 온도
        self.state = np.array([25.0, 40.0, 25.0, 0.0, 1000.0]) # 초기값

    def step(self, action):
        # 1. Action 적용 (간단한 물리 법칙 모사)
        heater_adj, rpm_adj = action
        
        curr_temp, humidity, ex_temp, air_press, rpm = self.state
        
        # 온도는 히터에 의해 오르고, RPM(풍량)에 의해 일부 냉각됨을 가정
        new_temp = curr_temp + (heater_adj * 5.0) - (rpm * 0.001)
        new_rpm = np.clip(rpm + (rpm_adj * 100), 500, 3000)
        
        # 나머지 변수들도 물리 공식에 따라 업데이트 (여기서는 단순화)
        self.state = np.array([new_temp, humidity, new_temp-5, air_press, new_rpm])
        
        # 2. Reward 계산: 목표 온도와의 차이가 작을수록 높은 보상
        reward = -abs(self.target_temp - new_temp)
        
        # 3. 종료 조건
        terminated = bool(new_temp < 0 or new_temp > 150)
        truncated = False
        
        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        self.state = np.array([25.0, 40.0, 25.0, 10.0, 1000.0])
        return self.state, {}


### 2. 강화 학습 모델 학습 및 실행

Stable Baselines3를 사용하여 에이전트를 학습시킵니다.

In [4]:
from stable_baselines3 import PPO

# 환경 생성
env = DryingOvenEnv()
print(f'env is defined.[{env}]')


# 모델 정의 (MlpPolicy: 다층 퍼셉트론 신경망)
model = PPO("MlpPolicy", env, verbose=1)
print(f'mode is defined. [{model}]')
# 학습 시작
model.learn(total_timesteps=10000)

print(f'model is learned. [{model}]')

test_cnt = 10

# 학습된 모델로 제어 테스트
obs, _ = env.reset()
for _ in range(10):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"현재 온도: {obs[0]:.2f}°C, 팬 RPM: {obs[4]:.2f}")
    if terminated: break


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 32        |
|    ep_rew_mean     | -1.94e+03 |
| time/              |           |
|    fps             | 3137      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 18.4        |
|    ep_rew_mean          | -1.17e+03   |
| time/                   |             |
|    fps                  | 151         |
|    iterations           | 2           |
|    time_elapsed         | 27          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011365855 |
|    clip_fraction        | 0.166       |
|    clip_range           | 0.2         |
|    entro

### **핵심 포인트**

- State 설정: 내부 온습도뿐만 아니라 배기 온도와 풍압을 넣음으로써 외부 열 손실이나 공기 흐름의 변화를 모델이 학습할 수 있게 합니다.
- Reward 설계: 단순히 온도 차이뿐만 아니라, 에너지 효율(RPM 최소화)이나 급격한 온도 변화 방지를 보상 함수에 추가하면 더 안정적인 제어가 가능합니다.
- 실제 적용: 실제 오븐에 연결할 때는 위 시뮬레이터 대신 PLC나 센서 데이터(MQTT/Modbus)를 받아오는 인터페이스 코드로 교체해야 합니다.